# 05. HTTP, REST APIs, and async programming

**Level:** Beginner to intermediate

**Prerequisites:** 03-04: functions, exceptions, JSON.

[Course index](00_START_HERE.ipynb) | [Previous](04_Files_JSON_and_Data_Validation.ipynb) | [Next](06_Environments_Pip_and_Jupyter.ipynb)

**Topics:** HTTP, REST APIs, Async programming, Timeouts, Mock HTTP.

**Matching script:** `examples\05_HTTP_REST_and_Async.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

HTTP is a request/response protocol: a request has a method, path, headers, and possibly a body.
Responses have status codes, headers, and a body. GET conventionally retrieves; POST often
submits work or creates a resource. JSON is one body representation, not HTTP itself.

REST describes architectural constraints around resource-oriented interaction; not every JSON
endpoint is a fully RESTful design. An inference endpoint commonly accepts validated features
and returns a prediction plus model version. A 200 response does not prove the prediction is
correct; schema validation and model evaluation remain separate responsibilities.

async def defines a coroutine, and await cooperatively waits. Blocking I/O inside async def
still blocks the event loop. Use an async client with explicit timeouts, bound concurrency,
handle non-success status codes, and avoid logging credentials. Retrying a write may duplicate
side effects; use a documented idempotency policy.

The exercise uses httpx.MockTransport. No request leaves the process, no server is started,
and the hostname is a synthetic .test domain. A notebook uses top-level await; a standalone
script normally calls asyncio.run(main()) at its entry point.

## Mathematics or data contract

Concurrency overlaps waiting; it does not automatically parallelize CPU-heavy model training.

## Important limitation

The mock validates client behavior only, not a real endpoint, network, TLS, or authentication.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Async JSON requests entirely in memory

**Purpose:** Exercise the HTTP boundary without credentials or network access.

### Read the code in this order
1. Implement a deterministic request handler.
2. Send through MockTransport with an explicit timeout.
3. Check status before reading JSON.
4. Gather two results and validate order and prediction values.

**Notebook/script difference:** this cell uses top-level `await main()` because Jupyter already has an event loop. Its `.py` counterpart calls `asyncio.run(main())` once. The operation and assertions are the same.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `async` defines cooperative operations; `await` waits without starting a separate CPU process.
- `with`/`async with` bounds a resource lifetime or a temporary execution mode.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
mock HTTP labels: [0, 1]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import asyncio
import json
import httpx

def handler(request):
    assert request.method == "POST" and request.url.path == "/predict"
    record = json.loads(request.content)
    return httpx.Response(200, json={"label": int(record["latency_ms"] >= 20)})

async def main():
    transport = httpx.MockTransport(handler)
    async with httpx.AsyncClient(transport=transport, timeout=2.0) as client:
        async def predict(value):
            response = await client.post("https://model.test/predict", json={"latency_ms": value})
            response.raise_for_status()
            return response.json()["label"]
        labels = await asyncio.gather(predict(5), predict(25))
    assert labels == [0, 1]
    print("mock HTTP labels:", labels)

await main()

mock HTTP labels: [0, 1]


### Interpret example 1

The result should demonstrate this purpose: **Exercise the HTTP boundary without credentials or network access.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Make the mock return HTTP 503 and assert HTTPStatusError rather than accepting an empty prediction.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `HTTP` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Concurrency overlaps waiting; it does not automatically parallelize CPU-heavy model training.
- Name the boundary of the demonstration: The mock validates client behavior only, not a real endpoint, network, TLS, or authentication.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://www.python-httpx.org/async/`
- `https://www.python-httpx.org/advanced/transports/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.